# CRYSTALS-Kyber

In the world of cryptography the prospect of practical quantum computers also comes with great threats to the security of our communication protocols. When coupled with [Shor's algorithm](https://en.wikipedia.org/wiki/Shor%27s_algorithm), the current Public-Key Encryption (PKE) schemes become virtually obsolete.

Asymmetric cryptography relies on hard mathematical problems to render brute-force attacks impractical. The factoring problem of RSA, the discrete logarithm of Diffie-Hellman are some examples of the "hard" problems that Shor's algorithm is set to solve in polynomial time on a large enough quantum computer.

Post-Quantum Cryptography is therefore a class of asymmetric encryption schemes designed around "problems" able to retain their hardness in the face of a quantum attacker.

The first algorithm to be standardized by NIST is [CRYSTALS-Kyber](https://pq-crystals.org/kyber/), a lattice-based Key Encapsulation Mechanism (KEM). A KEM is a key-exchange protocol built upon a PKE scheme. Kyber's (and almost all post-quantum algorithms') PKE is only CPA secure (i.e. Chosen-Plaintext Attack secure), lacking ciphertext integrity. Therefore, the encryption is used to *encapsulate* a shared secret (a symmetric key) that is sent to the other communication party, which *decapsulates* it.

To decapsulate the shared secret while offering ciphertext integrity Kyber uses a method called the Fujisaki-Okamoto Transform: the ciphertext is first decrypted using the private (PKE) key, and then re-encrypted with the public key. If the re-encrypted ciphertext matches the received one, the shared secret is computed from it, otherwise the exchange is terminated.

# Correlation Power Analysis

Side-Channel Attacks are attacks that exploit both design and implementation details of algorithms that enable uncontrollable data leakage. The data is not leaked directly, but through related information (side-channels) such as time, memory access patterns, or power usage.

The power consumption leakage lets the adversary distinguish between instructions (e.g. load and store operations might consume more power than arithmetic operations) and between different data fed to those instructions. The attack that relies on this side-channel is generally known as Power Analysis.

Correlation Power Analysis (CPA) is a method that creates a theoretical model of the power consumption and correlates it with the actual measurements (called power traces). The most common model used is the Hamming weight (i.e. the number of '1' bits in the data being processed by the target function). The metric for correlation is the [Pearson Correlation Coefficient](https://en.wikipedia.org/wiki/Pearson_correlation_coefficient).

Using CPA an attacker would generally perform the following steps in order to retrieve the secret key:

0.   Analyze the target algorithm and find a computation that might leak information about the secret key (i.e. a computation involving the secret key and attacker-controlled/chosen data);
1.   Choose or generate a vector of *controlled* data to be used in the next computations;
2.   Acquire a dataset of power traces by probing the target algorithm while it runs the target function/computation with every data point generated at *step 1* (see the [Chipwhisperer](https://www.newae.com/chipwhisperer) boards);
3.   Compute the target function's results using the generated data vector and all possible values for the secret key (the secret key candidates) - that is for each secret key candidate the attacker will compute a vector of target results;
4.   Compute the Hamming weights of the results from *step 3* - there will be a vector of Hamming weights for each secret key candidate;
4.   Compute the correlation between all Hamming weight vectors and all traces and pick the highest correlation coefficient;
5.   Pick the secret key candidate corresponding to the Hamming weight vector with the highest correlation as the correct secret key.

# CPA Attack on Kyber

In this lab we will perform a CPA attack on the first step of the polynomial multiplication from Kyber's decryption method.

When receiving a ciphertext from Alice during a key exchange, Bob calls the decapsulation method to retrieve the shared secret. The first step is to decrypt the ciphertext using the PKE private/secret key, which is our target as attackers.

During decryption the ciphertext polynomials are multiplied to the secret key polynomials in the Number Theoretic Transform domain (NTT). NTT could be seen as a DFT (Discrete Fourier Transform) for finite fields/modulo operations and is used as an optimization for polynomial multiplication. Kyber specifically performs these multiplications in pairs of coefficients like so ($u_i$ and $s_i$ are the i-th coefficients of the ciphertext and secret key respectively, $z$ is a constant (representing a root of unity), and Q is the modulo constant, which for Kyber is equal to 3329):

\begin{equation}
  (u_0, u_1) \cdot (s_0, s_1) = (r_0, r_1)
\end{equation}

\begin{equation}
  r_0 = (u_1 \cdot s_1 \cdot z + u_0 \cdot s_0)\ mod\ Q
\end{equation}

\begin{equation}
  r_1 = (u_1 \cdot s_0 + u_0 \cdot s_1)\ mod\ Q
\end{equation}

The Kyber algorithm implementation we will be targetting is from the [pqm4](https://github.com/mupq/pqm4) open-source library. The project aims to provide implementations for post-quantum algorithms optimized for the ARM Cortex M4 processor, which is one of the most commonly used in side-channel analysis. Among pqm4's particularities are the use of signed integers and Plantard modular reduction, and packing two 16-bit results into a single 32-bit register, while still performing the computations on 16-bit halfwords.

In the `plain_traces.npy` file we have 2000 power traces corresponding to the execution of `PKE_decrypt` with 2000 random ciphertexts (given in NTT form in `ntt_target_results.txt`). The secret key that we'll try to find using CPA is found in `secret_key.txt`.

The traces captured leakage from the first polynomial multiplication of the ciphertext $u$ and secret key $s$, so the first 3 steps of the attack presented before are already completed.

More details about the attack can be found in [this paper](https://www.mdpi.com/2410-387X/9/1/19).

# Attack Preparation



In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import multiprocessing as mp
import time

## Constants

In [ ]:
CHUNK_SIZE = 32
MAX_TRACES_PER_FILE = 2000
KYBER_K = 2
KYBER_Q = 3329
KYBER_QA = 26632
KYBER_Q_INV = 0x6ba8f301          # -3327
KYBER_N = 256
MONT_R_INV = 169
BARRET_CONST = 20159
KYBER_POLYBYTES = 384
KYBER_POLYVECBYTES = KYBER_K * KYBER_POLYBYTES
CIPHERTEXT_BYTES = KYBER_K * 320 + 128
PUBLIC_KEY_BYTES = KYBER_K * KYBER_POLYBYTES + 32
SECRET_KEY_BYTES = KYBER_K * KYBER_POLYBYTES + PUBLIC_KEY_BYTES + 2 * 32

zetas = [
    21932846, 3562152210, 752167598, 3417653460, 2112004045, 932791035, 2951903026, 1419184148, 1817845876, 3434425636,
    4233039261, 300609006, 975366560, 2781600929, 3889854731, 3935010590, 2197155094, 2130066389, 3598276897, 2308109491,
    2382939200, 1228239371, 1884934581, 3466679822, 1211467195, 2977706375, 3144137970, 3080919767, 945692709, 3015121229,
    345764865, 826997308, 2043625172, 2964804700, 2628071007, 4154339049, 483812778, 3288636719, 2696449880, 2122325384,
    1371447954, 411563403, 3577634219, 976656727, 2708061387, 723783916, 3181552825, 3346694253, 3617629408, 1408862808,
    519937465, 1323711759, 1474661346, 2773859924, 3580214553, 1143088323, 2221668274, 1563682897, 2417773720, 1327582262,
    2722253228, 3786641338, 1141798155, 2779020594
]

## Helper Functions

In [ ]:
def unpack_ciphertext(ct, i):
    poly_coeffs = [0] * KYBER_N

    for j in range(KYBER_N // 4):
        poly_coeffs[4*j+0] =  (((ct[320*i+5*j+0]       | ((ct[320*i+5*j+1] & 0x03) << 8)) * KYBER_Q) + 512) >> 10
        poly_coeffs[4*j+1] = ((((ct[320*i+5*j+1] >> 2) | ((ct[320*i+5*j+2] & 0x0f) << 6)) * KYBER_Q) + 512) >> 10
        poly_coeffs[4*j+2] = ((((ct[320*i+5*j+2] >> 4) | ((ct[320*i+5*j+3] & 0x3f) << 4)) * KYBER_Q) + 512) >> 10
        poly_coeffs[4*j+3] = ((((ct[320*i+5*j+3] >> 6) | ((ct[320*i+5*j+4] & 0xff) << 2)) * KYBER_Q) + 512) >> 10

    return poly_coeffs

def get_halfword(num: int, half):
    if half == 't':
        num = num >> 16
    elif half == 'b':
        num = num & 0x0000ffff
    else:
        print('Error: invalid halfword suffix! Returning number unchanged.')

    # Make sure to keep the numbers signed
    num = num.to_bytes(3, byteorder='little', signed=True)
    num = int.from_bytes(num[:2], byteorder='little', signed=True)

    return num

def poly_reduce(poly: list):
    for i in range(KYBER_N // 2):
        tmp = poly[i] * BARRET_CONST                # smulbb \tmp, \a, \barrettconst
        tmp2 = poly[i+1] * BARRET_CONST             # smultb \tmp2, \a, \barrettconst
        tmp = tmp >> 26                             # asr \tmp, \tmp, #26
        tmp2 = tmp2 >> 26                           # asr \tmp2, \tmp2, #26
        tmp = get_halfword(tmp, 'b') * KYBER_Q      # smulbb \tmp, \tmp, \q
        tmp2 = get_halfword(tmp2, 'b') * KYBER_Q    # smulbb \tmp2, \tmp2, \q
        poly[i] -= tmp                              # pkhbt \tmp, \tmp, \tmp2, lsl#16
        poly[i+1] -= tmp2                           # usub16 \a, \a, \tmp

    return poly

def poly_tobytes(poly: list):
    coeffs = poly_reduce(poly)
    poly_bytes = bytearray(3 * (KYBER_N // 2))

    for i in range(KYBER_N // 2):
        t0 = coeffs[2 * i]
        t1 = coeffs[2 * i + 1]
        poly_bytes[3 * i] = t0 & 0xff
        poly_bytes[3 * i + 1] = (t0 >> 8) | ((t1 & 0xf) << 4)
        poly_bytes[3 * i + 2] = (t1 >> 4) & 0xff

    return poly_bytes

def poly_frombytes(poly_bytes: bytearray):
    coeffs = [0] * KYBER_N

    for i in range(KYBER_N // 2):
        coeffs[2 * i] = poly_bytes[3 * i] | ((poly_bytes[3 * i + 1] & 0x0f) << 8)
        coeffs[2 * i + 1] = (poly_bytes[3 * i + 1] >> 4) | ((poly_bytes[3 * i + 2] & 0xff) << 4)

    return coeffs

def polyvec_tobytes(polyvec: list):
    polyvec_bytes = b''

    for poly in polyvec:
        polyvec_bytes += poly_tobytes(poly)

    return polyvec_bytes

def polyvec_frombytes(polyvec_bytes: bytearray):
    vec = []

    for i in range(KYBER_K):
        start = i * KYBER_POLYBYTES
        vec.append(poly_frombytes(polyvec_bytes[start:]))

    return vec

## Task 1 - Visualize Power Traces (1p)

Load the ciphertext, secret key, and power traces and plot the first power trace. Analyze the power consumption leakage. What can be observed? What insights could be drawn by looking at it?

In [ ]:
def load_traces(traces_file, ciphertexts_file, secret_key_file, ntraces) -> "tuple[list[list], list[int], list[int]]":
    combined_traces = np.load(traces_file)
    traces = [trace.tolist() for trace in combined_traces]

    with open(secret_key_file, 'r') as sk_file:
        secret_key = eval(sk_file.readline().replace('bytearray', '').replace('(', '[').replace(')', ']'))[0]
        secret_key += eval(sk_file.readline().replace('bytearray', '').replace('(', '[').replace(')', ']'))[0]

    with open(ciphertexts_file, 'r') as f:
        ntt_cts, mul_res = ''.join(f.readlines()).replace('ntt_cts: ', '').split('\nmul_res:')
        ntt_cts = ntt_cts.replace('\n\t', ',')
        mul_res = mul_res.replace('\n\t', ',')
        ntt_ciphertexts = eval('[' + ntt_cts + ']')
        # ntt_mul_results = eval('[' + mul_res + ']')

    sk_polyvec = polyvec_frombytes(secret_key[:SECRET_KEY_BYTES])
    sk_first_poly = sk_polyvec[0]

    return traces[:ntraces], ntt_ciphertexts, sk_first_poly

In [ ]:
ntraces = 2000
traces_file = 'plain_traces.npy'
ciphertexts_file = 'ntt_target_results.txt'
secret_key_file = 'secret_key.txt'

# TODO: plot and analyze a power trace

## Task 2 - Hamming Weight (1.5p)

Write a function to compute the Hamming weight of a 16-bit number. Remember that the pqm4 Kyber implementation packs two 16-bit numbers in 32-bit registers, but the computations are performed on halfwords.

In [ ]:
def hamming_weight(num):
    # TODO: compute the Hamming weight of `num`
    pass

## Task 3 - Basemul (2p)

Write a function to compute the first target coefficient ($r_0$) of the basemul operation. The function should translate the assembly instructions into python code. You can use the `get_halfword` helper function to select the bottom or top of a 32-bit register. Use the Cortex M4 [instruction set manual](https://www.st.com/resource/en/programming_manual/pm0214-stm32-cortexm4-mcus-and-mpus-programming-manual-stmicroelectronics.pdf) to determine the correct operations.
    
    .macro doublebasemul_frombytes_asm rptr, bptr, zeta, poly0, poly1, poly3,tmp,  tmp2, q, qa, qinv
    ldr.w \poly0, [\bptr], #4

    smulwt \tmp, \zeta, \poly1
    smlabt \tmp, \tmp, \q, \qa  
    smultt \tmp, \poly0, \tmp  
    smlabb \tmp, \poly0, \poly1, \tmp
    // a1*b1*zeta+a0*b0
    plant_red \q, \qa, \qinv, \tmp
    // r[0] in upper half of tmp
    ...
  
For example the `smulwt` operation is a signed multiplication word by halfword: the first term is the result register, followed by the two 32-bit operands: using the first in full, and only the top part of the second. The result should also be on 32 bits by choosing the most significant ones:

    smulwt \tmp, \zeta, \poly1 <=>
    tmp = smulwt(zeta, poly1) = zeta * get_halfword(poly1, 't') <=>
    tmp = (zeta * poly1[16:]) >> 16           # result on 48 bits, return the most significant 32 (thus ">> 16")

Here `poly0` packs the two ciphertext coefficients `(u0, u1)`, while `poly1` packs the two secret key coefficients `(s0, s1)` used in the computations. Therefore, the above code should be interpreted as: `tmp = (zeta * s1) >> 16`.


In [ ]:
def compute_r0(u0, u1, s0, s1, zeta):
  r0 = (zeta * s1) >> 16                             # smulwt(zeta, poly1)           result on 48 bits, return the most significant 32 (thus >> 16)

  # TODO: implement the other instructions of the basemul function

  # TODO 1: smlabt(tmp, q, qa)
  # TODO 2: smultt(poly0, tmp)
  # TODO 3: smlabb(poly0, poly1, tmp)
  # TODO 4: mul(a, q_inv)
  # TODO 5: smlatt(tmp, q, qa)
  # TODO 6: result in high half

  return r0

# Attack Phases

Normally, a CPA attack is performed in one go, by computing the correlation between all traces and the Hamming weights of all results obtained from the chosen ciphertexts and each secret key candidate. This, however, could take quite a long time to test.

In the following tasks we will streamline the attack execution by simulating having a clone device on which we could set a known secret key and find intermediary results/information.

# Task 4 - Attack Phase 1 (2p)

First, we will determine the sample point (point in time) in a power trace where our target function (basemul) is executed.

To do that, write a function that takes in two secret key coefficients, computes the basemul results using all the corresponding ciphertext coefficients, determines the Hamming weights of those results, and then computes the Pearson Correlation Coefficient between the Hamming weights and the power traces. It should return the sample point/index with the highest correlation.

Note:
* `traces[i]` is the power trace recorded for the execution with `ciphertexts[i]`; `traces[i][j]` is the power measurement at sample point `j` for `traces[i]`;
* `ciphertexts[i][j]` is the j-th coefficient of the i-th ciphertext

In [ ]:
def cpa_attack_phase1(traces, ciphertexts, known_s0, known_s1, coeff_ind=0):
  corrs = []
  H = []
  zeta = zetas[coeff_ind // 4] * (-1 if (coeff_ind // 2) % 2 == 1 else 1)

  # TODO: implement the first phase of the attack: given (s0, s1) and (u0, u1), compute all r0 and correlate their Hamming weights to the power traces

  return corrs

In [ ]:
known_s0 = known_s[0]
known_s1 = known_s[1]
ind_max_corr = -1

corrs = cpa_attack_phase1(traces, ciphertexts, known_s0, known_s1, 0)

# TODO: print the found sample point and plot the correlations for each sample point

# Task 5 - Attack Phase 2 (2p)

Now that we know where to look in the power trace, it's time to retrieve the secret key coefficients. To work faster, we fix the first coefficient and search for the second one.

The phase 2 function will be fairly similar to the one from the first phase. This time you should compute the basemul results using the ciphertext coefficients, the known secret coefficient, and all possible values for the second one. The Hamming weights of these results should be correlated to the power measurements at the previously found sample point.

In [ ]:
def cpa_attack_phase2(traces, ciphertexts, known_s0, known_sample_point, coeff_ind=0):
  # TODO: implement the second phase of the attack: given (u0, u1), the first secret coefficient s0, and the found sample point, compute the correlations for all possible key candidates for s1
  pass

In [ ]:
corrs = cpa_attack_phase2(traces, ciphertexts, known_s0, ind_max_corr)

# TODO: print the found key and plot the correlations for all key candidates

# Task 6 - Finding Both Secret Coefficients (1.5p)

Having done that, let's now try to search for both secret coefficients at the same time. Implement the `cpa_attack_full` function so that you'll find the pair of coefficients with the highest correlation.

Since both coefficients are in `[0, KYBER_Q]` and the attack runs with `ntraces` ciphertexts and traces, the asymptotic complexity would be $O(Q^2 * ntraces)$. This could take a while to run, so is there a way to optimize it? If you can't find a way to run over the entire search space, try to at least vary the first secret coefficient as much as you can (instead of having it "known" as previously).

In [ ]:
def cpa_attack_full(traces, ciphertexts, coeff_ind, known_sample_point):
    # TODO: implement a version of the phase 2 attack that varies the first secret coefficient as well
    pass

In [ ]:
# TODO: you might need to use a reduced set of traces for this task; try to not go below 50
ntraces = 200
traces = traces[:ntraces]
ciphertexts = ciphertexts[:ntraces]
corrs = cpa_attack_full(traces, ciphertexts, 0, ind_max_corr)

# TODO: print the found secret coefficients and plot the correlations of all key candidates

# Task 7 (Bonus) - Minimum Number of Traces (1p)

Now we've seen that the attack works and is able to retrieve the secret key coefficients, one additional thing we could do is find the minimum number of traces needed for the attack to work. Running the attack multiple times with a growing number of traces and plotting the correlation coefficient for the correct key alongside other "promising" key candidates should give us a pretty good idea of what that optimal number is.

For this task use either the phase 2 or the full attack function and plot a subset of key candidates (the first N with the biggest correlation coefficient) alongside the correct key's correlations for a few different numbers of traces. You should obtain a plot that shows how the incorrect keys' correlations drop towards 0 as the number of traces increases, while the correct key retains its correlation coefficient high.

In [ ]:
# TODO: choose some checkpoints for which to verify if the attack succeeds
checkpoints = []

fig = plt.figure(figsize=(20, 10))
ax = fig.gca()
ax.set_xlabel('Number of traces')
ax.set_ylabel('Correlation')
ax.set_title('Evolution of correlation with the number of traces')
ax.set_xticks(checkpoints)

# TODO: run the attack and plot the evolution of correlation over an increasing number of traces for the correct key and some key candidates